<img src="https://upload.wikimedia.org/wikipedia/commons/3/35/Uba_fiuba_ingenieria_logo.png" width="300" align="center">



# **Analisis de Series de Tiempo II**

# **Clase 4, Embeddings Categóricos**

Silenciamos los mensajes informativos de TensorFlow

In [ ]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

Importamos las librerias necesarias

In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

Desactivamos avisos secundarios

In [ ]:
tf.get_logger().setLevel("ERROR")

Fijamos las semillas para reproducibilidad

In [ ]:
np.random.seed(42)
keras.utils.set_random_seed(42)

Descargamos el panel de desempleo por industria

In [ ]:
url = "https://raw.githubusercontent.com/vega/vega-datasets/main/data/unemployment-across-industries.json"
df = pd.read_json(url)
panel = df.pivot_table(index="date", columns="series", values="count")

In [ ]:
panel.head()

series,Agriculture,Business services,Construction,Education and Health,Finance,Government,Information,Leisure and hospitality,Manufacturing,Mining and Extraction,Other,Self-employed,Transportation and Utilities,Wholesale and Retail Trade
date,,,,,,,,,,,,,,
2000-01-01 08:00:00+00:00,154.0,655.0,745.0,353.0,228.0,430.0,125.0,782.0,734.0,19.0,274.0,239.0,236.0,1000.0
2000-02-01 08:00:00+00:00,173.0,587.0,812.0,349.0,240.0,409.0,112.0,779.0,694.0,25.0,232.0,262.0,223.0,1023.0
2000-03-01 08:00:00+00:00,152.0,623.0,669.0,381.0,226.0,311.0,140.0,789.0,739.0,17.0,247.0,213.0,192.0,983.0
2000-04-01 08:00:00+00:00,135.0,517.0,447.0,329.0,197.0,269.0,95.0,658.0,736.0,20.0,240.0,218.0,191.0,793.0
2000-05-01 07:00:00+00:00,73.0,561.0,397.0,423.0,195.0,370.0,131.0,675.0,685.0,27.0,254.0,206.0,190.0,821.0


Guardamos la lista de industrias

In [ ]:
industrias = list(panel.columns)
N_SERIES = len(industrias)

Definimos los hiperparametros del experimento

In [ ]:
L = 12
N_TEST = 24
UNIDADES = 16

Definimos la dimension del embedding

In [ ]:
DIM_EMB = 4

Mapeamos cada nombre de industria a un entero: este es el input categorico

El entero no tiene significado ordinal, es solo la fila de la tabla de embeddings

In [ ]:
id_industria = {ind: i for i, ind in enumerate(industrias)}
print("Codificacion de industrias:")
for ind, i in list(id_industria.items())[:4]:
    print(f"  {i}: {ind}")
print(f"  ... hasta {N_SERIES - 1}")

Codificacion de industrias:
  0: Agriculture
  1: Business services
  2: Construction
  3: Education and Health
  ... hasta 13


Creamos las listas del pool de entrenamiento y los diccionarios de test

In [ ]:
Xp, yp, idp = [], [], []
X_te, y_te, id_te, escalas = {}, {}, {}, {}

Recorremos cada industria construyendo sus ventanas

In [ ]:
for ind in industrias:

    # Extraemos la serie y la estandarizamos con estadisticas de entrenamiento
    v = panel[ind].values.astype("float32")
    mu, sd = v[:-N_TEST].mean(), v[:-N_TEST].std()
    escalas[ind] = (mu, sd)
    z = (v - mu) / sd

    # Construimos las ventanas deslizantes
    Xs, ys = [], []
    for t in range(L, len(z)):
        Xs.append(z[t - L:t].reshape(-1, 1))
        ys.append(z[t])

    Xs = np.array(Xs, dtype="float32")
    ys = np.array(ys, dtype="float32").reshape(-1, 1)

    # Contamos cuantas ventanas de entrenamiento aporta esta industria
    n_train = len(Xs) - N_TEST

    # Acumulamos el tramo de entrenamiento en el pool global
    Xp.append(Xs[:-N_TEST])
    yp.append(ys[:-N_TEST])

    # Guardamos el identificador repetido una vez por ventana de esta industria
    idp.append(np.full((n_train,), id_industria[ind], dtype="int32"))

    # Guardamos el test de esta industria por separado, con su identificador
    X_te[ind] = Xs[-N_TEST:]
    y_te[ind] = ys[-N_TEST:]
    id_te[ind] = np.full((N_TEST,), id_industria[ind], dtype="int32")

Concatenamos el pool completo de entrenamiento

In [ ]:
X_pool = np.concatenate(Xp, axis=0)
y_pool = np.concatenate(yp, axis=0)
id_pool = np.concatenate(idp, axis=0)
print(f"\nPool global: {X_pool.shape[0]} ventanas de {N_SERIES} industrias")


Pool global: 1204 ventanas de 14 industrias


### **A) Global**

Definimos el modelo global sin ninguna nocion de identidad

In [ ]:
entrada_a = keras.Input(shape=(L, 1), name="ventana")
h_a = layers.LSTM(UNIDADES)(entrada_a)
salida_a = layers.Dense(1)(h_a)
modelo_anonimo = keras.Model(entrada_a, salida_a, name="global_anonimo")

Compilamos y entrenamos sobre el pool

In [ ]:
modelo_anonimo.compile(optimizer=keras.optimizers.Adam(0.01), loss="mse")
print(f"Parametros modelo anonimo: {modelo_anonimo.count_params():,}")
modelo_anonimo.fit(X_pool, y_pool, epochs=100, batch_size=64, verbose=0)

Parametros modelo anonimo: 1,169


Evaluamos por industria y desestandarizamos el error

In [ ]:
mae_anonimo = {}
for ind in industrias:
    pred = modelo_anonimo.predict(X_te[ind], verbose=0)
    mae_anonimo[ind] = np.abs(pred - y_te[ind]).mean() * escalas[ind][1]

### **B) Global + Embedding**

Definimos dos entradas: la ventana numerica y el identificador categorico

In [ ]:
entrada_serie = keras.Input(shape=(L, 1), name="ventana")
entrada_id = keras.Input(shape=(1,), dtype="int32", name="id_industria")

Definimos la capa de embedding: una tabla de N_SERIES filas x DIM_EMB columnas


Es una lookup table diferenciable, equivalente a una capa lineal sobre el one-hot

In [ ]:
capa_emb = layers.Embedding(
    input_dim=N_SERIES,      # Cantidad de categorias distintas
    output_dim=DIM_EMB,      # Dimension del vector denso por categoria
    name="embedding_industria",
)

Aplicamos el embedding

In [ ]:
emb = capa_emb(entrada_id)

Repetimos el vector de identidad en los L pasos temporales de la ventana


El embedding es estatico (no cambia con t), pero la secuencia si: hay que replicarlo

In [ ]:
emb_repetido = layers.Lambda(
    lambda x: tf.repeat(x, repeats=L, axis=1),
    output_shape=(L, DIM_EMB),
)(emb)

Concatenamos la serie con su identidad: el input pasa de 1 a 1+DIM_EMB canales

In [ ]:
entrada_combinada = layers.Concatenate(axis=2, name="serie_mas_identidad")(
    [entrada_serie, emb_repetido]
)

Aplicamos el LSTM sobre la secuencia enriquecida con la identidad

In [ ]:
h_b = layers.LSTM(UNIDADES, name="lstm_con_identidad")(entrada_combinada)

Proyectamos a la prediccion escalar

In [ ]:
salida_b = layers.Dense(1, name="prediccion")(h_b)

Ensamblamos el modelo con sus dos entradas

In [ ]:
modelo_emb = keras.Model([entrada_serie, entrada_id], salida_b, name="global_con_embedding")

Compilamos: el embedding se entrena por backpropagation junto al resto de la red

In [ ]:
modelo_emb.compile(optimizer=keras.optimizers.Adam(0.01), loss="mse")
print(f"Parametros modelo con embedding: {modelo_emb.count_params():,}")

Parametros modelo con embedding: 1,481


Entrenamos con las dos entradas en simultaneo

In [ ]:
modelo_emb.fit(
    [X_pool, id_pool.reshape(-1, 1)], y_pool,
    epochs=100, batch_size=64, verbose=0,
)

Evaluamos por industria pasando tambien el identificador

In [ ]:
mae_emb = {}
for ind in industrias:
    pred = modelo_emb.predict(
        [X_te[ind], id_te[ind].reshape(-1, 1)], verbose=0
    )
    mae_emb[ind] = np.abs(pred - y_te[ind]).mean() * escalas[ind][1]

Imprimimos la comparacion por industria

In [ ]:
print(f"{'Industria':<30}{'Anonimo':>12}{'Embedding':>12}{'Gana':>10}")
print("_" * 64)
for ind in sorted(industrias, key=lambda s: mae_emb[s]):
    ganador = "emb" if mae_emb[ind] < mae_anonimo[ind] else "ANONIMO"
    print(f"{ind:<30}{mae_anonimo[ind]:>12.0f}{mae_emb[ind]:>12.0f}{ganador:>10}")
print("-" * 64)

prom_an = np.mean(list(mae_anonimo.values()))
prom_em = np.mean(list(mae_emb.values()))
print(f"{'PROMEDIO':<30}{prom_an:>12.0f}{prom_em:>12.0f}")
print("=" * 64)

Industria                          Anonimo   Embedding      Gana
________________________________________________________________
Mining and Extraction                   34          30       emb
Information                             44          48   ANONIMO
Agriculture                             49          50   ANONIMO
Other                                  108          99       emb
Transportation and Utilities           166         159       emb
Government                             161         163   ANONIMO
Finance                                177         177       emb
Self-employed                          180         193   ANONIMO
Education and Health                   285         281       emb
Business services                      308         292       emb
Leisure and hospitality                364         341       emb
Manufacturing                          452         385       emb
Wholesale and Retail Trade             399         388       emb
Construction             

Interpretamos el resultado en terminos de capacidad vs evidencia

In [ ]:
gana_emb = sum(1 for i in industrias if mae_emb[i] < mae_anonimo[i])
print(f"\nEl embedding gana en {gana_emb} de {N_SERIES} industrias.")


El embedding gana en 9 de 14 industrias.


Calculamos la mejora porcentual del promedio

In [ ]:
mejora = (1 - prom_em / prom_an) * 100
mejora

np.float32(3.1152546)

1. La mejora promedio es marginal 3.11 y no es uniforme. El embedding pierde en 5 industrias. Con 86 ventanas por serie la identidad apenas tiene respaldo estadistico para justificarse.

2. Esto es el hilo de la clase por tercera vez: toda capacidad adicional es una hipotesis que los datos deben poder pagar.

3. El embedding le devuelve al modelo la capacidad de memorizar por serie que el pooling le habia quitado: deshace parte de la regularizacion implicita que hacia funcionar al modelo global.

### **C) Lo que aprendió el embedding**

Extraemos la matriz de embeddings entrenada: forma (N_SERIES, DIM_EMB)

In [ ]:
E = capa_emb.get_weights()[0]

Normalizamos cada vector para calcular similitud coseno

In [ ]:
E_norm = E / np.linalg.norm(E, axis=1, keepdims=True)

Calculamos la matriz de similitud coseno entre todas las industrias

In [ ]:
similitud = E_norm @ E_norm.T

Mostramos los vecinos mas cercanos de tres industrias de interes

In [ ]:
for ref in ["Construction", "Finance", "Manufacturing"]:

    # Recuperamos la fila de similitudes de la industria de referencia
    fila = similitud[id_industria[ref]]

    # Ordenamos de mayor a menor y descartamos la primera (es ella misma)
    orden = np.argsort(-fila)[1:4]
    vecinos = ", ".join(f"{industrias[j]} ({fila[j]:.2f})" for j in orden)
    print(f"{ref:<18}: {vecinos}")

Construction      : Government (0.80), Agriculture (0.78), Wholesale and Retail Trade (0.08)
Finance           : Leisure and hospitality (0.67), Other (0.23), Education and Health (0.20)
Manufacturing     : Information (0.87), Wholesale and Retail Trade (0.76), Other (0.74)


Nadie declaro estas similitudes: dos series con dinamicas parecidas generan gradientes parecidos sobre sus embeddings, asi que sus vectores se mueven juntos. Usos operativos: diagnostico de segmentacion, cold start de series nuevas, y deteccion de series mal clasificadas en la taxonomia de negocio.